### CNN for image classification using CIFAR10 datasets


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision      # Specific for CV
from torchvision.datasets import CIFAR10

#### torchvision  ----> CV
#### datasets     ----> CIFAR10, MNIST
#### pretrainesd CNNs
#### utilities for image transformation

In [2]:
# Datasets and dataloader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale (0,1) => normalize => (-1, 1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

D:\Anaconda 2\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [3]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [4]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

### Building CNN


In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  #kernal size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  #kernal size = 2, stride = 2

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  #kernel size = 2, stride = 2
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)      # Flattening
        x = self.fc_layers(x)

        return x

In [6]:
model = CNN()

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [8]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images)    #Forward propagation
        loss = criterion(output, labels)   # loss function
        loss.backward()   # Backward propagation
        optimizer.step()    #update parameters

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=1.3613894104652697
epoch=2/10 & loss=0.9210921129607179
epoch=3/10 & loss=0.7327295907623018
epoch=4/10 & loss=0.6057097577606626
epoch=5/10 & loss=0.5004866449805476
epoch=6/10 & loss=0.40445051468013193
epoch=7/10 & loss=0.3220566693512375
epoch=8/10 & loss=0.23923144089367687
epoch=9/10 & loss=0.1880310737739896
epoch=10/10 & loss=0.14951967434181124


### Model Evaluation

In [9]:
correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {correct_labels / total_labels *100}")

Accuracy = 75.82
